In [11]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd

service = Service(ChromeDriverManager().install())
options = Options()
options.add_argument("--headless")
options.add_argument("--window-size=1920x1080")
options.add_argument("--lang=ko_KR")
options.add_argument("--start-maximized")
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
options.add_argument(f"--user-agent={user_agent}")

driver = webdriver.Chrome(service = service, options = options)
url = "https://tickets.interpark.com/contents/genre/concert?app_header_state=hide"
driver.get(url)
time.sleep(2)

try :
    hot_button = driver.find_element(By.XPATH, "//button[contains(text(), '요즘 HOT')]")
    print(hot_button)
    hot_button.click()
    time.sleep(2)
    
except Exception as e:
    print("요즘 HOT 버튼 클릭 실패 :" , e)

scroll_pause = 1
last_height = driver.execute_script("return document.body.scrollHeight")

for _ in range(5) :
    new_height = driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(scroll_pause)
    if new_height == last_height :
        break
    last_height = new_height

titles = driver.find_elements(By.CSS_SELECTOR, ".TicketItem_goodsName__Ju76j")
dates = driver.find_elements(By.CSS_SELECTOR, ".TicketItem_playDate__5ePr2")
places = driver.find_elements(By.CSS_SELECTOR, ".TicketItem_placeName__ls_9C")

data = []

for t, d, p in zip(titles, dates, places) :
    data.append({
        "title" : t.text.strip(),
        "date" : d.text.strip(),
        "place" : p.text.strip()
    })

driver.quit()

df = pd.DataFrame(data)
df.head() #시각화

<selenium.webdriver.remote.webelement.WebElement (session="753b47f31db0f7f112c9e16cad0ff90f", element="f.2484165FFE89FA79133165FEB9EB47B9.d.6B1B33539EDC2F8B3459BE4CC3F5F10E.e.38")>
[{'title': '변진섭 전국투어 콘서트 : 변천사 시즌2', 'date': '2025.02.09 -', 'place': '전국 각 지역'}, {'title': 'RETRO MUSIC CONCERT 〈최재훈×플라워〉', 'date': '2025.08.09, 2025.10.19', 'place': '서울,인천'}, {'title': '옥탑재즈', 'date': '2025.02.14 - 2025.10.24', 'place': '아트센터인천 다목적홀'}, {'title': '2025 심수봉 전국투어 콘서트：꽃길 - 안동', 'date': '2025.09.13', 'place': '안동문화예술의전당 웅부홀'}, {'title': '홍경민 매달리스트 콘서트', 'date': '2025.08.02', 'place': 'SA HALL'}]


,title,date,place
0,변진섭 전국투어 콘서트 : 변천사 시즌2,2025.02.09 -,전국 각 지역
1,RETRO MUSIC CONCERT 〈최재훈×플라워〉,"2025.08.09, 2025.10.19","서울,인천"
2,옥탑재즈,2025.02.14 - 2025.10.24,아트센터인천 다목적홀
3,2025 심수봉 전국투어 콘서트：꽃길 - 안동,2025.09.13,안동문화예술의전당 웅부홀
4,홍경민 매달리스트 콘서트,2025.08.02,SA HALL


In [12]:
pip install pymysql

Note: you may need to restart the kernel to use updated packages.


In [17]:
import pymysql

conn = pymysql.connect(
    host = 'localhost',
    user = 'root',
    password ='Vmfpswm163@@',
    db = 'interpark',
    charset = 'utf8mb4'
)

cursor = conn.cursor()
#cursor() : DB 내 테이블에 값을 넣어줄 수 있는 요소

#iterrows() : 추출해온 데이터에서 하나의 행을 반복적으로 찾아오는 함수
# -, 인덱스값을 찾아오는데 그 인덱스값을 쓰지 않기 위해
for _, row in df.iterrows() :
    sql = """ 
    INSERT INTO performances (title, date_range, place)
    VALUES (%s, %s, %s)
    """
    # """ """ > 문장을 다 인식함
    # %s : 형식지정자, 순서를 유지하여 값을 넣는다
    cursor.execute(sql, (row["title"], row["date"], row["place"]))

conn.commit() #DB에 반영해달라는 명령어
conn.close()